# IKG ODM Insight Lineage Explorer

Traces column-level lineage for any ODM **insight type** back to source tables.

**Workflow:**
1. Run cells 1–5 (imports → connect → load)
2. Type to search an insight type in the searchable dropdown (Cell 6)
3. Click **▶ Trace Lineage** → writes `<insight_type>_lineage.html` next to this notebook
4. Open the HTML file in Chrome / Edge / Firefox


## 1. Imports

In [ ]:
import pandas as pd
import json
import os
import re
import datetime
from pathlib import Path
from collections import deque
from IPython.display import display as ipy_display, clear_output
import ipywidgets as widgets


## 2. Configuration

In [ ]:
# ── Greenplum connection ─────────────────────────────────────────────
GP_HOST = 'greenplum-rdsp.zur.swissbank.com'
GP_PORT = 5432
GP_DB   = 'gprdsp'
GP_USER = 'ds_rdsp_dev'
GP_PASSWORD = ''  # leave blank to be prompted

# ── Schema / table names ─────────────────────────────────────────────
IKG_SCHEMA = 'sandbox_prj_smart_insights'
ODM_SCHEMA = 'core_ikg'
LIN_TABLE  = 'ikg_column_lineage_master_auto_refresh'
ODM_TABLE  = 'odm_rule_metadata_auto_refresh'


## 3. Connect to Greenplum

In [ ]:
import getpass, sqlalchemy
if not GP_PASSWORD:
    GP_PASSWORD = getpass.getpass('Greenplum password: ')
engine = sqlalchemy.create_engine(
    f'postgresql+psycopg2://{GP_USER}:{GP_PASSWORD}@{GP_HOST}:{GP_PORT}/{GP_DB}'
)
print('Connected to Greenplum')


## 4. Load Data

In [ ]:
# ── Lineage table ────────────────────────────────────────────────────
df_lin = pd.read_sql(
    f'SELECT * FROM {IKG_SCHEMA}.{LIN_TABLE}', engine
).fillna('')
print(f'Lineage: {len(df_lin):,} rows')

# ── Insight types from ODM metadata ──────────────────────────────────
df_it = pd.read_sql(
    f'SELECT DISTINCT insight_type FROM {ODM_SCHEMA}.{ODM_TABLE}'
    ' WHERE insight_type IS NOT NULL ORDER BY insight_type',
    engine
)
IT_LIST = [it for it in df_it['insight_type'].tolist() if it and str(it).strip()]
print(f'Insight types: {len(IT_LIST)}')


## 5. Helper Functions

In [ ]:
# ── Profile-row predicate ────────────────────────────────────────────
def is_profile(r):
    tt  = str(r.get('target_table',     '') or '').lower().strip()
    stt = str(r.get('sub_target_table', '') or '').lower().strip()
    return tt == stt and tt.endswith('_profile_curr_ikg')

_rows = df_lin.to_dict('records')

def _get(tbl, col):
    t, c = tbl.lower(), col.lower()
    return [r for r in _rows
            if str(r.get('sub_target_table','') or '').lower() == t
            and str(r.get('target_column',   '') or '').lower() == c]

def _schema(tbl):
    t = tbl.lower()
    for r in _rows:
        if str(r.get('source_table',     '') or '').lower() == t and r.get('source_schema'):    return r['source_schema']
        if str(r.get('sub_target_table', '') or '').lower() == t and r.get('sub_target_schema'): return r['sub_target_schema']
    return ''

def trace(col, tbl):
    nid = lambda a, b: f'{a.lower()}::{b.lower()}'
    nodes, nrows, edges, visited = {}, {}, {}, set()
    q = deque()

    def add(t, c, sc, start=False):
        k = nid(t, c)
        if k not in nodes:
            nodes[k]  = {'id': k, 'tbl': t, 'col': c, 'schema': sc, 'is_start': start}
            nrows[k]  = []
        return k

    seed = [r for r in _rows if is_profile(r)
            and str(r.get('target_table',  '') or '').lower() == tbl.lower()
            and str(r.get('target_column', '') or '').lower() == col.lower()]
    if not seed: return None

    sid = add(tbl, col, seed[0].get('target_schema', ''), True)
    nrows[sid] = seed
    visited.add(sid)
    for r in seed:
        q.append((r.get('source_table', '') or '', r.get('source_column', '') or '', sid, [r]))

    itr = 0
    while q and itr < 800:
        itr += 1
        stbl, scol, pid, tr = q.popleft()

        if not stbl and scol:
            up = [r for r in _rows if is_profile(r)
                  and str(r.get('target_column', '') or '').lower() == scol.lower()]
            for r in up:
                uid = add(r['target_table'], scol, r.get('target_schema', ''))
                ek  = f'{uid}>{pid}'
                if ek not in edges: edges[ek] = {'from': uid, 'to': pid}
                nrows[uid].append(r)
                if uid not in visited:
                    visited.add(uid)
                    sub = _get(r['target_table'], scol)
                    nrows[uid].extend(sub)
                    for sr in sub:
                        q.append((sr.get('source_table','') or '', sr.get('source_column','') or '', uid, [sr]))
            continue

        if not stbl and not scol: continue

        sc_val = _schema(stbl)
        src_id = add(stbl, scol, sc_val)
        ek     = f'{src_id}>{pid}'
        if ek not in edges: edges[ek] = {'from': src_id, 'to': pid}
        if src_id in visited: continue
        visited.add(src_id)
        sub = _get(stbl, scol)
        nrows[src_id].extend(sub)
        for sr in sub:
            q.append((sr.get('source_table','') or '', sr.get('source_column','') or '', src_id, [sr]))

    return {'nodes': list(nodes.values()), 'edges': list(edges.values()), 'nrows': nrows}

def safe_name(s): return re.sub(r'[^a-zA-Z0-9_-]', '_', str(s).strip())

print('Helpers ready')


## 6. HTML Template

In [ ]:
# Store the HTML template in a Python variable
HTML_TPL = r'''<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<title>__INSIGHT__ — IKG Lineage</title>
<script src="https://cdnjs.cloudflare.com/ajax/libs/sigma.js/2.4.0/sigma.min.js"></script>
<script src="https://cdnjs.cloudflare.com/ajax/libs/graphology/0.25.4/graphology.umd.min.js"></script>
<style>
*{box-sizing:border-box;margin:0;padding:0;}
body{font-family:"Inter","Segoe UI",Arial,sans-serif;height:100vh;display:flex;flex-direction:column;overflow:hidden;background:#f4f7fb;}
#topbar{background:linear-gradient(120deg,#0d1f3c 0%,#1a3a6e 55%,#1565c0 100%);color:#fff;padding:10px 22px;display:flex;align-items:center;gap:14px;box-shadow:0 2px 14px rgba(0,0,0,.32);flex-shrink:0;z-index:20;}
#topbar h1{font-size:16px;font-weight:800;letter-spacing:.3px;white-space:nowrap;}
#topbar h1 span{color:#90caf9;}
#topbar .sep{width:1px;height:26px;background:rgba(255,255,255,.2);flex-shrink:0;}
#topbar .meta{font-size:12px;color:rgba(255,255,255,.62);}
#topbar .meta b{color:rgba(255,255,255,.92);font-weight:700;}
#ts{font-size:11px;color:rgba(255,255,255,.38);margin-left:auto;white-space:nowrap;}
#main{display:flex;flex:1;overflow:hidden;}
/* GRAPH */
#gwrap{flex:1;position:relative;background:linear-gradient(150deg,#ecf1fb 0%,#e4ecf8 100%);overflow:hidden;}
#sc{width:100%;height:100%;}
/* CAMERA */
#cam{position:absolute;bottom:18px;right:18px;display:flex;flex-direction:column;gap:6px;}
.cbtn{width:34px;height:34px;background:#fff;border:1.5px solid #b8cde8;color:#1a3a6e;border-radius:8px;cursor:pointer;font-size:18px;font-weight:800;display:flex;align-items:center;justify-content:center;box-shadow:0 2px 8px rgba(0,0,0,.12);transition:all .15s;}
.cbtn:hover{background:#1565c0;color:#fff;border-color:#1565c0;box-shadow:0 3px 10px rgba(21,101,192,.35);}
/* LEGEND */
#legend{position:absolute;bottom:18px;left:18px;background:rgba(255,255,255,.94);border:1px solid #d0daf0;border-radius:10px;padding:11px 15px;box-shadow:0 3px 14px rgba(0,0,0,.1);max-width:230px;backdrop-filter:blur(6px);}
#legend h4{font-size:10px;font-weight:800;text-transform:uppercase;letter-spacing:.6px;color:#4a6080;margin-bottom:8px;}
.lr{display:flex;align-items:center;gap:9px;margin-bottom:5px;font-size:12px;color:#263248;}
.ld{width:14px;height:14px;border-radius:4px;flex-shrink:0;border:2px solid rgba(0,0,0,.15);}
/* TOOLTIP */
#tip{position:absolute;background:#1a2744;color:#fff;padding:8px 13px;border-radius:8px;font-size:12px;display:none;pointer-events:none;white-space:nowrap;z-index:30;box-shadow:0 4px 16px rgba(0,0,0,.28);}
#tip .tt{font-weight:700;font-size:13px;color:#90caf9;letter-spacing:.2px;}
#tip .tc{color:rgba(255,255,255,.68);font-size:11px;margin-top:2px;}
/* LOADING */
#loader{position:absolute;inset:0;background:rgba(244,247,251,.78);display:none;align-items:center;justify-content:center;z-index:50;flex-direction:column;gap:14px;}
#loader.on{display:flex;}
.spin{width:44px;height:44px;border:4px solid #c0cfe8;border-top-color:#1565c0;border-radius:50%;animation:rot .75s linear infinite;}
@keyframes rot{to{transform:rotate(360deg);}}
#loader p{color:#2e4a6e;font-size:14px;font-weight:500;}
/* RIGHT PANEL */
#rpanel{width:340px;background:#fff;border-left:1.5px solid #d4deee;display:flex;flex-direction:column;flex-shrink:0;overflow:hidden;}
#rphdr{background:linear-gradient(90deg,#0d1f3c,#1a3a6e);color:#fff;padding:11px 16px;font-size:13px;font-weight:700;letter-spacing:.2px;flex-shrink:0;display:flex;align-items:center;gap:8px;}
#rpbody{flex:1;overflow-y:auto;padding:14px;}
.placeholder{display:flex;flex-direction:column;align-items:center;justify-content:center;height:100%;gap:10px;color:#7a92b0;text-align:center;padding:30px;}
.placeholder .icon{font-size:44px;opacity:.45;}
.placeholder p{font-size:13px;line-height:1.6;}
/* DETAIL CARDS */
.ncard{margin-bottom:10px;border:1.5px solid #d4deee;border-radius:10px;overflow:hidden;box-shadow:0 1px 6px rgba(0,0,0,.05);}
.ncard-hdr{padding:9px 13px;font-size:12px;font-weight:700;display:flex;align-items:center;gap:8px;flex-wrap:wrap;}
.ncard-tbl{font-size:14px;font-weight:800;color:#fff;letter-spacing:.2px;}
.ncard-col{font-size:12px;color:rgba(255,255,255,.72);font-weight:500;font-style:italic;}
.nrow{display:flex;padding:5px 13px;border-bottom:1px solid #eef2f9;font-size:13px;}
.nrow:last-child{border-bottom:none;}
.nlbl{color:#607090;width:120px;flex-shrink:0;font-size:11.5px;font-weight:600;letter-spacing:.1px;}
.nval{color:#1a2744;word-break:break-word;font-weight:500;}
.nval.empty{color:#b0bec8;font-style:italic;font-weight:400;}
.nval.mono{font-family:"Courier New",monospace;font-size:11px;background:#f2f6fc;padding:2px 6px;border-radius:4px;color:#1a3a6e;}
.rbadge{display:inline-flex;align-items:center;padding:2px 9px;border-radius:9px;font-size:11px;font-weight:700;letter-spacing:.3px;}
.rb-sel{background:#dbeafe;color:#1d4ed8;}
.rb-join{background:#d1fae5;color:#065f46;}
.rb-wh{background:#fef3c7;color:#92400e;}
.rb-hav{background:#fce7f3;color:#9d174d;}
.rb-val{background:#ede9fe;color:#5b21b6;}
.rb-str{background:#cffafe;color:#155e75;}
#rpbody::-webkit-scrollbar{width:5px;}
#rpbody::-webkit-scrollbar-thumb{background:#c0cfe8;border-radius:3px;}
</style>
</head>
<body>
<div id="topbar">
  <h1>IKG Lineage <span>Explorer</span></h1>
  <div class="sep"></div>
  <div class="meta">Insight: <b>__INSIGHT__</b></div>
  <div class="sep"></div>
  <div class="meta">Pairs: <b>__PAIRS__</b>&nbsp;&nbsp;Nodes: <b id="nc">—</b>&nbsp;&nbsp;Edges: <b id="ec">—</b></div>
  <div id="ts">__TS__</div>
</div>
<div id="main">
  <div id="gwrap">
    <div id="sc"></div>
    <div id="loader"><div class="spin"></div><p>Building lineage graph…</p></div>
    <div id="cam">
      <button class="cbtn" title="Zoom in"  onclick="camZ(1.35)">+</button>
      <button class="cbtn" title="Zoom out" onclick="camZ(0.74)">−</button>
      <button class="cbtn" title="Fit"      onclick="camFit()">⊡</button>
    </div>
    <div id="legend"><h4>Schema</h4><div id="leg"></div></div>
    <div id="tip"><div class="tt" id="tt-t"></div><div class="tc" id="tt-c"></div></div>
  </div>
  <div id="rpanel">
    <div id="rphdr">&#128202; Node Details</div>
    <div id="rpbody">
      <div class="placeholder"><div class="icon">&#128269;</div><p>Click any node to see<br>lineage details here.</p></div>
    </div>
  </div>
</div>
<script>
const GD=__GD__;
const PALETTE=[
  ["core_ikg",{bg:"#1565c0",br:"#0d47a1"}],
  ["ikg_schema",{bg:"#1565c0",br:"#0d47a1"}],
  ["sandbox_prj_smart_insights",{bg:"#1565c0",br:"#0d47a1"}],
  ["sandbox_ikg_pre_prd",{bg:"#1976d2",br:"#1565c0"}],
  ["core_wma_shared",{bg:"#00796b",br:"#004d40"}],
  ["edw_view_input_schema",{bg:"#00796b",br:"#004d40"}],
  ["ikg_vendor_schema",{bg:"#00796b",br:"#004d40"}],
  ["edw_input_schema",{bg:"#0288d1",br:"#01579b"}],
  ["core_wma_shared_masked",{bg:"#00897b",br:"#00695c"}],
  ["sandbox_wma_shared",{bg:"#388e3c",br:"#1b5e20"}],
  ["sandbox_prj_ds_data",{bg:"#f57c00",br:"#e65100"}],
  ["core_nlg",{bg:"#7b1fa2",br:"#4a148c"}],
  ["nlg_schema",{bg:"#7b1fa2",br:"#4a148c"}],
  ["core_model",{bg:"#5d4037",br:"#3e2723"}],
  ["model_schema",{bg:"#5d4037",br:"#3e2723"}],
  ["sandbox_prj_sbl",{bg:"#6a1b9a",br:"#4a148c"}],
  ["sandbox_prj_dsforoverdrive",{bg:"#0277bd",br:"#01579b"}],
  ["core_in_shared",{bg:"#6d4c41",br:"#4e342e"}],
  ["ikg_clip_schema",{bg:"#6d4c41",br:"#4e342e"}],
  ["sandbox_prj_adhoc",{bg:"#ad1457",br:"#880e4f"}],
  ["sandbox_prj_smart_relationship",{bg:"#00838f",br:"#006064"}],
  ["ikg_wealthx_schema",{bg:"#00838f",br:"#006064"}],
  ["sandbox_prj_rbat",{bg:"#558b2f",br:"#33691e"}],
];
const DEF={bg:"#455a64",br:"#263238"};
function theme(s){
  const sl=(s||"").toLowerCase().trim();
  for(const [k,v] of PALETTE) if(sl===k||sl.includes(k)||k.includes(sl)) return v;
  return DEF;
}
let sig=null,G=null;
function boot(){
  document.getElementById("loader").classList.add("on");
  setTimeout(()=>{ try{_build();}catch(ex){console.error(ex);} document.getElementById("loader").classList.remove("on"); },40);
}
function _build(){
  if(sig){sig.kill();sig=null;}
  G=new graphology.Graph({type:"directed",multi:false});
  const {nodes,edges,nrows}=GD;
  // Kahn topo sort
  const aO=new Map(),id_=new Map();
  nodes.forEach(n=>{aO.set(n.id,[]);id_.set(n.id,0);});
  edges.forEach(e=>{if(aO.has(e.from)&&aO.has(e.to)){aO.get(e.from).push(e.to);id_.set(e.to,(id_.get(e.to)||0)+1);}});
  const lv=new Map(),q=[];
  id_.forEach((d,id)=>{if(d===0)q.push(id);});
  while(q.length){const id=q.shift(),l=lv.get(id)||0;(aO.get(id)||[]).forEach(t=>{lv.set(t,Math.max(lv.get(t)||0,l+1));id_.set(t,id_.get(t)-1);if(id_.get(t)===0)q.push(t);});}
  nodes.forEach(n=>{if(!lv.has(n.id))lv.set(n.id,0);});
  const bL=new Map();
  nodes.forEach(n=>{const l=lv.get(n.id)||0;if(!bL.has(l))bL.set(l,[]);bL.get(l).push(n);});
  const maxL=Math.max(...bL.keys(),0);
  const W=1100,H=680,PAD=110;
  const xS=maxL>0?(W-PAD*2)/maxL:W/2;
  const schemas=new Map();
  bL.forEach((ns,l)=>{
    const x=PAD+(maxL-l)*xS;
    ns.forEach((n,i)=>{
      const y=(i+1)*(H/(ns.length+1));
      const th=theme(n.schema);
      const sk=(n.schema||"unknown").toLowerCase();
      if(!schemas.has(sk))schemas.set(sk,{label:n.schema||"unknown",bg:th.bg});
      G.addNode(n.id,{x,y,size:n.is_start?20:13,color:th.bg,
        label:n.tbl+"\n"+n.col,_tbl:n.tbl,_col:n.col,_schema:n.schema,_s:n.is_start});
    });
  });
  const eS=new Set();
  edges.forEach((e,i)=>{
    const k=e.from+">"+e.to;
    if(G.hasNode(e.from)&&G.hasNode(e.to)&&!eS.has(k)){
      eS.add(k);G.addEdge(e.from,e.to,{size:2.5,color:"rgba(80,120,200,0.50)",type:"arrow"});
    }
  });
  const c=document.getElementById("sc");
  sig=new Sigma(G,c,{
    renderEdgeLabels:false,defaultEdgeType:"arrow",
    labelFont:\'"Inter","Segoe UI",Arial\',labelWeight:"700",
    labelColor:{color:"#1a2744"},labelSize:11,
    labelDensity:1,labelGridCellSize:120,
    minCameraRatio:0.03,maxCameraRatio:15,
    nodeReducer:(node,data)=>({
      ...data,
      label:data._tbl+"\n"+data._col,
      borderColor:theme(data._schema).br,
    }),
  });
  sig.on("clickNode",({node})=>detail(node));
  const tip=document.getElementById("tip");
  sig.on("enterNode",({node,event})=>{
    const a=G.getNodeAttributes(node);
    document.getElementById("tt-t").textContent=a._tbl;
    document.getElementById("tt-c").textContent="column: "+a._col;
    tip.style.display="block";mv(event.original);
  });
  sig.on("leaveNode",()=>{tip.style.display="none";});
  c.addEventListener("mousemove",ev=>{if(tip.style.display!=="none")mv(ev);});
  function mv(ev){const r=c.getBoundingClientRect();tip.style.left=(ev.clientX-r.left+15)+"px";tip.style.top=(ev.clientY-r.top-10)+"px";}
  const lb=document.getElementById("leg");
  lb.innerHTML=[...schemas.values()].slice(0,12).map(({label,bg})=>
    `<div class="lr"><div class="ld" style="background:${bg}"></div><span>${e$(label)}</span></div>`
  ).join("");
  document.getElementById("nc").textContent=G.order;
  document.getElementById("ec").textContent=G.size;
  camFit();
}
function detail(nodeId){
  const {nrows}=GD;
  const rows=nrows[nodeId]||[];
  const a=G.getNodeAttributes(nodeId);
  const th=theme(a._schema);
  const body=document.getElementById("rpbody");
  const seen=new Set();
  const deduped=rows.filter(r=>{
    const k=r.sql_process+"|"+r.source_table+"|"+r.source_column+"|"+r.target_column;
    if(seen.has(k))return false;seen.add(k);return true;
  });
  let h=`<div class="ncard">
    <div class="ncard-hdr" style="background:${th.bg};border-bottom:3px solid ${th.br}">
      <div>
        <div class="ncard-tbl">&#128200; ${e$(a._tbl)}</div>
        <div class="ncard-col">column: ${e$(a._col)}</div>
      </div>
    </div>
    <div>
      ${nr("Schema",a._schema)}
      ${nr("Role",a._s?"&#9733; Profile Target":"Source / Intermediate")}
    </div>
  </div>`;
  if(!deduped.length){
    h+=`<div style="color:#7a92b0;font-size:13px;padding:18px;text-align:center;line-height:1.7">
      <div style="font-size:32px;margin-bottom:8px">&#128204;</div>
      Base source — no further upstream lineage.
    </div>`;
    body.innerHTML=h;return;
  }
  deduped.forEach((r,i)=>{
    const proc=r.sql_process||"select";
    h+=`<div class="ncard">
      <div class="ncard-hdr" style="background:${th.bg}22;border-bottom:2px solid ${th.br}33">
        ${bdg(proc)}
        <span style="font-size:11px;color:#4a6080;font-weight:600;margin-left:4px">Record ${i+1} / ${deduped.length}</span>
      </div>
      <div>
        ${nr("Target Table",r.target_table)}
        ${nr("Target Schema",r.target_schema)}
        ${nr("Sub-Target Table",r.sub_target_table)}
        ${nr("Sub-Target Schema",r.sub_target_schema)}
        ${nr("Target Column",r.target_column)}
        ${nr("Source Table",r.source_table)}
        ${nr("Source Schema",r.source_schema)}
        ${nr("Source Column",r.source_column)}
        ${nr("Process",r.process)}
        ${nr("SQL Process",r.sql_process)}
        ${r.logic?nrm("Logic",r.logic.slice(0,450)):""}
      </div>
    </div>`;
  });
  body.innerHTML=h;
}
function nr(l,v){const empty=!v||!String(v).trim();return `<div class="nrow"><span class="nlbl">${l}</span><span class="nval${empty?" empty":""}"> ${empty?"—":e$(String(v))}</span></div>`;}
function nrm(l,v){return `<div class="nrow"><span class="nlbl">${l}</span><span class="nval mono">${e$(String(v))}</span></div>`;}
function bdg(p){const m={"select":"rb-sel","select-value":"rb-val","select*":"rb-str","join":"rb-join","where":"rb-wh","having":"rb-hav","where-subquery":"rb-wh"};return `<span class="rbadge ${m[p]||"rb-sel"}">${e$(p||"select")}</span>`;}
function camZ(f){if(sig)sig.getCamera().animatedZoom({duration:200,factor:f});}
function camFit(){if(sig)sig.getCamera().animatedReset({duration:450});}
function e$(s){return String(s||"").replace(/&/g,"&amp;").replace(/</g,"&lt;").replace(/>/g,"&gt;").replace(/"/g,"&quot;");}
boot();
</script>
</body>
</html>
'''

print(f'Template ready ({len(HTML_TPL):,} chars)')


## 7. Interactive Explorer — select insight type and trace

In [ ]:
# ── Widgets ──────────────────────────────────────────────────────────
_stat = widgets.Output()

w_insight = widgets.Combobox(
    options       = IT_LIST,
    value         = '',
    placeholder   = 'Type to search insight type...',
    description   = 'Insight Type:',
    ensure_option = True,
    style         = {'description_width': '100px'},
    layout        = widgets.Layout(width='480px'),
)

btn = widgets.Button(
    description  = '\u25b6  Trace Lineage',
    button_style = 'primary',
    layout       = widgets.Layout(width='165px', height='36px'),
    disabled     = True,
)

def _on_change(change):
    btn.disabled = not bool((change['new'] or '').strip())
    with _stat:
        clear_output()
        if (change['new'] or '').strip():
            print(f'Selected: {change["new"]}  -- click Trace Lineage')
w_insight.observe(_on_change, names='value')

SAFE_FIELDS = [
    'target_table','target_schema','sub_target_table','sub_target_schema',
    'target_column','source_table','source_schema','source_column',
    'process','sql_process','logic'
]

def _on_trace(_):
    sel = (w_insight.value or '').strip()
    if not sel: return

    with _stat:
        clear_output()
        print(f'Loading ODM metadata for: {sel} ...')

    # Query ODM metadata
    q = ('SELECT DISTINCT profile_table, rule_column FROM '
         + ODM_SCHEMA + '.' + ODM_TABLE
         + " WHERE insight_type = %(sel)s"
         + " AND profile_table IS NOT NULL AND profile_table <> ''"
         + " AND rule_column   IS NOT NULL AND rule_column   <> ''")
    df_scope = pd.read_sql(q, engine, params={'sel': sel})

    if df_scope.empty:
        with _stat:
            clear_output()
            print(f'No metadata found for: {sel}')
        return

    with _stat:
        clear_output()
        print(f'{len(df_scope)} column-table pairs -- tracing lineage...')

    # Trace lineage for every (profile_table, rule_column) pair
    all_nodes, all_edges, all_nrows = {}, {}, {}
    for _, row in df_scope.iterrows():
        ptbl = str(row['profile_table']).strip()
        rcol = str(row['rule_column']).strip()
        res  = trace(rcol, ptbl)
        if not res: continue
        for n in res['nodes']:
            if n['id'] not in all_nodes: all_nodes[n['id']] = n
            elif n['is_start']:          all_nodes[n['id']]['is_start'] = True
        for e in res['edges']:
            ek = f'{e["from"]}>{e["to"]}'
            if ek not in all_edges: all_edges[ek] = e
        for k, rv in res['nrows'].items():
            all_nrows.setdefault(k, [])
            all_nrows[k].extend(rv)

    # Deduplicate nrows
    nrows_clean = {}
    for k, rv in all_nrows.items():
        seen_set, ded = set(), []
        for r in rv:
            rk = (r.get('sql_process',''), r.get('source_table',''),
                  r.get('source_column',''), r.get('target_column',''))
            if rk not in seen_set:
                seen_set.add(rk)
                ded.append({f: str(r.get(f,'') or '') for f in SAFE_FIELDS})
        nrows_clean[k] = ded

    nodes_list = list(all_nodes.values())
    edges_list = list(all_edges.values())
    if not nodes_list:
        with _stat:
            clear_output()
            print('No lineage nodes found.')
        return

    # Build + write HTML
    gd = {'nodes': nodes_list, 'edges': edges_list, 'nrows': nrows_clean}
    gd_json = json.dumps(gd, ensure_ascii=False)
    ts  = datetime.datetime.now().strftime('%Y-%m-%d %H:%M')
    out = (HTML_TPL
           .replace('__INSIGHT__', sel)
           .replace('__PAIRS__',   str(len(df_scope)))
           .replace('__TS__',      ts)
           .replace('__GD__',      gd_json))
    fname = safe_name(sel) + '_lineage.html'
    fpath = os.path.abspath(fname)
    with open(fname, 'w', encoding='utf-8') as f:
        f.write(out)
    with _stat:
        clear_output()
        print(f'{len(nodes_list)} nodes, {len(edges_list)} edges')
        print(f'Saved: {fpath}')
        print('Open in Chrome / Edge / Firefox.')

btn.on_click(_on_trace)
ipy_display(widgets.VBox([
    widgets.HBox([w_insight, btn]),
    _stat,
]))


## 7. Usage Guide

| Step | Action |
|------|--------|
| 1 | Run cells 1–5 in order |
| 2 | Run cell 6 to display the widget |
| 3 | **Type** to search/filter insight types in the dropdown |
| 4 | Click **▶ Trace Lineage** |
| 5 | Open `<insight_type>_lineage.html` in a browser |
| 6 | Click any node for full details in the right panel |

### Node colours (by schema)

| Schema | Colour |
|--------|--------|
| `core_ikg` / `sandbox_prj_smart_insights` | 🔵 Deep Blue |
| `core_wma_shared` / `edw_view_input_schema` | 🟢 Teal |
| `edw_input_schema` | 🔵 Sky Blue |
| `core_nlg` | 🟣 Purple |
| `core_model` | 🟤 Brown |
| `sandbox_prj_ds_data` | 🟠 Orange |
| `sandbox_wma_shared` | 🟢 Green |
| `sandbox_prj_smart_relationship` | 🩵 Cyan |

### Detail panel

Click any node to see: `process`, `target_table`, `target_schema`,
`sub_target_table`, `sub_target_schema`, `source_table`, `source_schema`,
`target_column`, `source_column`, `logic`, `sql_process`.
